In [5]:
"""
bilstm_bigru_ensemble_kaggle.py
==============================
Ensemble Model: (BiLSTM + Attention) & (BiGRU + Attention)
with Focal Loss + Offline Data Augmentation + GPU Acceleration

Run this on Kaggle (GPU T4x2 or GPU P100 runtime).

SETUP STEPS ON KAGGLE:
1. Click "+ Add Input" on the right panel.
2. Upload 'sliding_window_sequences.npz' and 'RVI_38_labels.mat' as a dataset.
3. Turn on GPU Accelerator (e.g., GPU T4 x2) in the notebook settings.
4. Run this script in a notebook cell:
   %run bilstm_bigru_ensemble_kaggle.py
"""

# ─── STEP 0: Environment & Path Detection ─────────────────────────────────────
import os, sys, glob

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'kaggle_web_client' in sys.modules or os.path.exists('/kaggle/input')

SEQ_PATH = None
LABEL_PATH = None
SAVE_DIR = None

if IN_KAGGLE:
    print("="*65)
    print("  [KAGGLE DETECTED] Scanning /kaggle/input recursively...")
    print("="*65)
    
    npz_files = glob.glob('/kaggle/input/**/*.npz', recursive=True)
    mat_files = glob.glob('/kaggle/input/**/*.mat', recursive=True)
    
    for f in npz_files:
        if 'sliding_window_sequences_448' in os.path.basename(f) or 'sliding_window_sequences' in os.path.basename(f):
            SEQ_PATH = f
            break
    for f in mat_files:
        if 'labels' in os.path.basename(f).lower() or 'rvi' in os.path.basename(f).lower():
            LABEL_PATH = f
            break
            
    if SEQ_PATH and LABEL_PATH:
        print(f"  ✓ Found sequences at: {SEQ_PATH}")
        print(f"  ✓ Found labels at: {LABEL_PATH}")
    else:
        print("  ⚠️ Could not auto-locate files. Defaulting to standard Kaggle path structure.")
        DRIVE_DIR = '/kaggle/input/cp-detection-dataset'
        SEQ_PATH = os.path.join(DRIVE_DIR, 'sliding_window_sequences.npz')
        LABEL_PATH = os.path.join(DRIVE_DIR, 'RVI_38_labels.mat')
        
    SAVE_DIR = '/kaggle/working'
    print(f"  ✓ Outputs will be saved to: {SAVE_DIR}")

elif IN_COLAB:
    print("="*65)
    print("  [COLAB DETECTED] Mounting Google Drive...")
    print("="*65)
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR  = '/content/drive/MyDrive/CP_Detection'
    SEQ_PATH   = os.path.join(DRIVE_DIR, 'sliding_window_sequences.npz')
    LABEL_PATH = os.path.join(DRIVE_DIR, 'RVI_38_labels.mat')
    SAVE_DIR   = DRIVE_DIR
else:
    print("="*65)
    print("  [LOCAL DETECTED] Using local paths...")
    print("="*65)
    BASE = r'c:\Users\Admin\Downloads\drive-download-20260601T063218Z-3-001'
    SEQ_PATH   = os.path.join(BASE, 'python', 'outputs', 'features', 'sliding_window_sequences.npz')
    LABEL_PATH = os.path.join(BASE, 'data', 'RVI_38_labels.mat')
    SAVE_DIR   = os.path.dirname(SEQ_PATH)

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import scipy.io
import time

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, backend as K
from sklearn.metrics import confusion_matrix

# ─── HYPERPARAMETERS ──────────────────────────────────────────────────────────
NUM_VIDEOS   = 38
N_FEATURES   = 448

# Model Dimensions
LSTM_UNITS   = 128      # GPU CuDNN optimized
DROPOUT      = 0.4
LEARN_RATE   = 5e-4

# Training
EPOCHS       = 100
BATCH_SIZE   = 16
PATIENCE     = 15

# Focal Loss params
FOCAL_GAMMA  = 2.0      # focuses on hard-to-learn CP examples
FOCAL_ALPHA  = 0.85     # heavy weight on CP class errors

# Augmentation copies
AUG_NOISE_STD    = 0.02
AUG_TIME_REVERSE = True
AUG_MIRROR       = True
AUG_N_COPIES     = 4      # 4 augmented versions per CP video

np.random.seed(42)
tf.random.set_seed(42)

# GPU config
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        gpu_msg = f"✓ GPU found ({len(gpus)} device(s)). Memory growth enabled."
    except RuntimeError as e:
        gpu_msg = f"⚠️ GPU config error: {e}"
else:
    gpu_msg = "⚠️ No GPU found. Running on CPU."

print("="*65)
print("  CP Detection Ensemble: BiLSTM + BiGRU + Temporal Attention")
print(f"  TF version: {tf.__version__}")
print(f"  {gpu_msg}")
print("="*65)


# ─── STEP 1: Load Data ────────────────────────────────────────────────────────
print("\n[1/5] Loading sequences...")
if not os.path.exists(SEQ_PATH) or not os.path.exists(LABEL_PATH):
    raise FileNotFoundError(
        f"Could not find files.\nSequences Path: {SEQ_PATH}\nLabels Path: {LABEL_PATH}\n"
        "Please ensure you uploaded the dataset and attached it to the notebook/session."
    )

data      = np.load(SEQ_PATH, allow_pickle=True)
sequences = [data[f'seq_{i}'] for i in range(NUM_VIDEOS)]

mat    = scipy.io.loadmat(LABEL_PATH)
labels = mat['labels'].flatten().astype(int)

print(f"  Loaded {len(sequences)} sequences")
print(f"  Labels: CP={labels.sum()} | Healthy={(labels==0).sum()}")
print(f"  Sequence lengths: min={min(s.shape[0] for s in sequences)} "
      f"max={max(s.shape[0] for s in sequences)}")
print(f"  Feature dim: {sequences[0].shape[1]}")

# ─── Fixed Class Weights (computed once on original dataset) ──────────────────
_n_total = len(labels)
_n_cp    = int(labels.sum())
_n_hlt   = _n_total - _n_cp

FIXED_CLASS_WEIGHTS = {
    0: _n_total / (2 * _n_hlt),
    1: _n_total / (2 * _n_cp)
}

print(f"\n  Fixed Class Weights (based on original {_n_total} videos):")
print(f"    Healthy (0) → {FIXED_CLASS_WEIGHTS[0]:.4f}")
print(f"    CP      (1) → {FIXED_CLASS_WEIGHTS[1]:.4f}")


# ─── STEP 2: Data Augmentation (Offline) ──────────────────────────────────────

def mirror_sequence(seq):
    """Swaps Left joint features with Right joint features in the 448-dim vector.
    Layout: G16(0-127) | B16(128-255) | A8(256-287) | H8(288-319) | C16(320-447)
    Each block: first half = Left joints, second half = Right joints
    """
    s = seq.copy()

    # G16: swap Left (0-63) ↔ Right (64-127)
    s[:, 0:64],    s[:, 64:128]   = seq[:, 64:128].copy(),   seq[:, 0:64].copy()

    # B16: swap Left (128-192) ↔ Right (192-256)
    s[:, 128:192], s[:, 192:256]  = seq[:, 192:256].copy(),  seq[:, 128:192].copy()

    # A8: LArm(256-264)↔RArm(272-280),  LLeg(264-272)↔RLeg(280-288)
    s[:, 256:264], s[:, 272:280]  = seq[:, 272:280].copy(),  seq[:, 256:264].copy()
    s[:, 264:272], s[:, 280:288]  = seq[:, 280:288].copy(),  seq[:, 264:272].copy()

    # H8: LW+LE(288-296)↔RW+RE(296-304),  LA+LK(304-312)↔RA+RK(312-320)
    s[:, 288:296], s[:, 296:304]  = seq[:, 296:304].copy(),  seq[:, 288:296].copy()
    s[:, 304:312], s[:, 312:320]  = seq[:, 312:320].copy(),  seq[:, 304:312].copy()

    # C16: swap Left (320-384) ↔ Right (384-448)
    s[:, 320:384], s[:, 384:448]  = seq[:, 384:448].copy(),  seq[:, 320:384].copy()

    return s


def add_noise(seq, std=AUG_NOISE_STD):
    return seq + np.random.normal(0, std, seq.shape).astype(np.float32)


def time_reverse(seq):
    return seq[::-1].copy()


def time_crop(seq, crop_ratio=0.85):
    n = seq.shape[0]
    keep = max(10, int(n * (crop_ratio + np.random.uniform(-0.05, 0.05))))
    start = np.random.randint(0, max(1, n - keep))
    return seq[start:start+keep]


def augment_cp_sequences(sequences, labels, n_copies=AUG_N_COPIES):
    aug_seqs  = list(sequences)
    aug_lbls  = list(labels)

    cp_indices = np.where(labels == 1)[0]
    for idx in cp_indices:
        seq = sequences[idx]

        # Copy 1: Mirror + Noise
        aug_seqs.append(add_noise(mirror_sequence(seq)))
        aug_lbls.append(1)

        # Copy 2: Time Reverse + Noise
        if n_copies >= 2:
            aug_seqs.append(add_noise(time_reverse(seq)))
            aug_lbls.append(1)

        # Copy 3: Mirror + Time Reverse + Noise
        if n_copies >= 3:
            aug_seqs.append(add_noise(mirror_sequence(time_reverse(seq))))
            aug_lbls.append(1)

        # Copy 4: Time Crop + Noise
        if n_copies >= 4:
            aug_seqs.append(add_noise(time_crop(seq)))
            aug_lbls.append(1)

    aug_lbls = np.array(aug_lbls)
    return aug_seqs, aug_lbls


print("\n[2/5] Data Augmentation strategy configured.")


# ─── STEP 3: Models & Attention ───────────────────────────────────────────────

def focal_loss(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA):
    def loss_fn(y_true, y_pred):
        y_pred   = K.clip(y_pred, K.epsilon(), 1.0 - K.epsilon())
        bce      = -y_true * K.log(y_pred) - (1 - y_true) * K.log(1 - y_pred)
        p_t      = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        alpha_t  = y_true * alpha + (1 - y_true) * (1 - alpha)
        fl_weight = alpha_t * K.pow(1.0 - p_t, gamma)
        return K.mean(fl_weight * bce)
    loss_fn.__name__ = f'focal_loss_g{gamma}_a{alpha}'
    return loss_fn


class AttentionLayer(layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.W     = layers.Dense(units, use_bias=False)
        self.V     = layers.Dense(1, use_bias=False)

    def call(self, hidden_states, mask=None):
        score = self.V(K.tanh(self.W(hidden_states)))  # (batch, time, 1)
        if mask is not None:
            mask_exp  = tf.cast(tf.expand_dims(mask, -1), tf.float32)
            score    += (1.0 - mask_exp) * (-1e9)
        attn_weights = K.softmax(score, axis=1)       # (batch, time, 1)
        context      = attn_weights * hidden_states   # (batch, time, features)
        context      = K.sum(context, axis=1)         # (batch, features)
        return context, attn_weights

    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units})
        return config


def build_bilstm_model(n_feat=N_FEATURES):
    """Model A: BiLSTM + Attention (GPU CuDNN compatible)"""
    inp  = keras.Input(shape=(None, n_feat), name='input')
    x    = layers.Masking(mask_value=0.0, name='masking')(inp)

    # BiLSTM 1 (no recurrent_dropout for GPU)
    x    = layers.Bidirectional(
               layers.LSTM(LSTM_UNITS, return_sequences=True, dropout=DROPOUT),
               name='bilstm_1')(x)
    x    = layers.LayerNormalization(name='ln_1')(x)

    # BiLSTM 2
    lstm2_out = layers.Bidirectional(
                    layers.LSTM(LSTM_UNITS // 2, return_sequences=True, dropout=DROPOUT),
                    name='bilstm_2')(x)
    x = layers.LayerNormalization(name='ln_2')(lstm2_out)

    # Attention Context
    context, attn_w = AttentionLayer(LSTM_UNITS, name='attention')(x)
    last_h = layers.Lambda(lambda t: t[:, -1, :], name='last_hidden')(x)
    combined = layers.Concatenate(name='combine')([context, last_h])

    # Head
    z = layers.Dense(64, activation='relu', name='dense_1')(combined)
    z = layers.Dropout(DROPOUT, name='drop_1')(z)
    z = layers.Dense(32, activation='relu', name='dense_2')(z)
    z = layers.Dropout(DROPOUT / 2, name='drop_2')(z)

    out = layers.Dense(1, activation='sigmoid', name='output')(z)

    model = keras.Model(inputs=inp, outputs=out, name='BiLSTM_Attention')
    model.compile(
        optimizer=keras.optimizers.Adam(LEARN_RATE, clipnorm=1.0),
        loss=focal_loss(FOCAL_GAMMA, FOCAL_ALPHA),
        metrics=['accuracy']
    )
    return model


def build_bigru_model(n_feat=N_FEATURES):
    """Model B: BiGRU + Attention (GPU CuDNN compatible)"""
    inp  = keras.Input(shape=(None, n_feat), name='input')
    x    = layers.Masking(mask_value=0.0, name='masking')(inp)

    # BiGRU 1 (no recurrent_dropout for GPU)
    x    = layers.Bidirectional(
               layers.GRU(LSTM_UNITS, return_sequences=True, dropout=DROPOUT),
               name='bigru_1')(inp) # Bypass raw masking if GRU handles it, but Masking is fine
    x    = layers.LayerNormalization(name='ln_1')(x)

    # BiGRU 2
    gru2_out = layers.Bidirectional(
                    layers.GRU(LSTM_UNITS // 2, return_sequences=True, dropout=DROPOUT),
                    name='bigru_2')(x)
    x = layers.LayerNormalization(name='ln_2')(gru2_out)

    # Attention Context
    context, attn_w = AttentionLayer(LSTM_UNITS, name='attention')(x)
    last_h = layers.Lambda(lambda t: t[:, -1, :], name='last_hidden')(x)
    combined = layers.Concatenate(name='combine')([context, last_h])

    # Head
    z = layers.Dense(64, activation='relu', name='dense_1')(combined)
    z = layers.Dropout(DROPOUT, name='drop_1')(z)
    z = layers.Dense(32, activation='relu', name='dense_2')(z)
    z = layers.Dropout(DROPOUT / 2, name='drop_2')(z)

    out = layers.Dense(1, activation='sigmoid', name='output')(z)

    model = keras.Model(inputs=inp, outputs=out, name='BiGRU_Attention')
    model.compile(
        optimizer=keras.optimizers.Adam(LEARN_RATE, clipnorm=1.0),
        loss=focal_loss(FOCAL_GAMMA, FOCAL_ALPHA),
        metrics=['accuracy']
    )
    return model


print("\n[3/5] Ensemble Architectures (BiLSTM & BiGRU) loaded.")


# ─── STEP 4: Normalization & Padding Helpers ──────────────────────────────────

def normalize(train_seqs, test_seq):
    cat   = np.vstack(train_seqs).astype(np.float32)
    mu    = cat.mean(0)
    sigma = cat.std(0)
    sigma[sigma == 0] = 1.0
    n_tr  = [(s.astype(np.float32) - mu) / sigma for s in train_seqs]
    n_te  = (test_seq.astype(np.float32) - mu) / sigma
    return n_tr, n_te


def pad_batch(seqs, labels):
    max_l = max(s.shape[0] for s in seqs)
    n_f   = seqs[0].shape[1]
    X     = np.zeros((len(seqs), max_l, n_f), dtype=np.float32)
    for i, s in enumerate(seqs):
        X[i, :s.shape[0]] = s
    return X, np.array(labels, dtype=np.float32)


# ─── STEP 5: LOOCV with Ensemble ──────────────────────────────────────────────

def run_ensemble_loocv(sequences, labels):
    n          = len(sequences)
    ens_probs  = np.zeros(n)
    lstm_probs = np.zeros(n)
    gru_probs  = np.zeros(n)
    t0         = time.time()

    for i in range(n):
        print(f"\n{'═'*65}")
        print(f"  Fold {i+1:02d}/{n}  |  Test = Video {i+1}  "
              f"|  Label = {'★ CP' if labels[i]==1 else 'Healthy'}")
        print(f"{'═'*65}")

        # Split
        tr_seqs_orig = [sequences[j] for j in range(n) if j != i]
        tr_lbls_orig = np.array([labels[j] for j in range(n) if j != i])
        te_seq       = sequences[i]
        te_lbl       = labels[i]

        # Augment CP sequences
        tr_seqs_aug, tr_lbls_aug = augment_cp_sequences(
            tr_seqs_orig, tr_lbls_orig, n_copies=AUG_N_COPIES)

        # Normalize
        tr_norm, te_norm = normalize(tr_seqs_aug, te_seq)

        # Pad & Batch
        X_tr, y_tr = pad_batch(tr_norm, tr_lbls_aug)
        X_te        = te_norm[np.newaxis].astype(np.float32)

        fit_callbacks = [
            keras.callbacks.EarlyStopping(
                monitor='loss', patience=PATIENCE,
                restore_best_weights=True, verbose=0),
            keras.callbacks.ReduceLROnPlateau(
                monitor='loss', factor=0.5, patience=7,
                min_lr=1e-6, verbose=0)
        ]

        # 1. Train Model A: BiLSTM
        print("  -> Training Model A (BiLSTM)...")
        model_lstm = build_bilstm_model()
        model_lstm.fit(
            X_tr, y_tr, epochs=EPOCHS, batch_size=BATCH_SIZE,
            class_weight=FIXED_CLASS_WEIGHTS, verbose=0, callbacks=fit_callbacks
        )
        p_lstm = float(model_lstm.predict(X_te, verbose=0)[0][0])
        lstm_probs[i] = p_lstm
        keras.backend.clear_session()

        # 2. Train Model B: BiGRU
        print("  -> Training Model B (BiGRU)...")
        model_gru = build_bigru_model()
        model_gru.fit(
            X_tr, y_tr, epochs=EPOCHS, batch_size=BATCH_SIZE,
            class_weight=FIXED_CLASS_WEIGHTS, verbose=0, callbacks=fit_callbacks
        )
        p_gru = float(model_gru.predict(X_te, verbose=0)[0][0])
        gru_probs[i] = p_gru
        keras.backend.clear_session()

        # 3. Weighted Ensemble (GRU-biased for higher CP sensitivity)
        # LSTM tends to be overconfident toward Healthy → downweight it
        LSTM_W, GRU_W = 0.3, 0.7
        p_ens = LSTM_W * p_lstm + GRU_W * p_gru
        ens_probs[i] = p_ens

        pred_ens = 1 if p_ens >= 0.5 else 0
        correct  = pred_ens == te_lbl
        eta      = (time.time() - t0) / (i + 1) * (n - i - 1)

        print(f"  LSTM(×{LSTM_W}) Prob={p_lstm:.3f} | GRU(×{GRU_W}) Prob={p_gru:.3f} | Ensemble Prob={p_ens:.3f}")
        print(f"  Pred Ensemble={'CP' if pred_ens==1 else 'Healthy'}  "
              f"True={'CP' if te_lbl==1 else 'Healthy'}  "
              f"{'✓ OK' if correct else '✗ WRONG'}  "
              f"|  ETA: {eta/60:.1f}min")

    return ens_probs, lstm_probs, gru_probs


print(f"\n[4/5] Running Improved Ensemble LOOCV...")
ens_probs, lstm_probs, gru_probs = run_ensemble_loocv(sequences, labels)


# ─── STEP 6: Results & Threshold Tuning ──────────────────────────────────────

def compute_and_print_metrics(y_true, probs, threshold=0.5, label=""):
    y_pred = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    acc  = (tp+tn)/len(y_true)*100
    sens = tp/(tp+fn)*100  if (tp+fn) > 0 else 0.
    spec = tn/(tn+fp)*100  if (tn+fp) > 0 else 0.
    prec = tp/(tp+fp)*100  if (tp+fp) > 0 else 0.
    f1   = 2*prec*sens/(prec+sens) if (prec+sens) > 0 else 0.
    den  = (tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)
    mcc  = (tp*tn-fp*fn)/den**0.5*100 if den > 0 else 0.

    print(f"\n{'═'*65}")
    print(f"  RESULTS: {label} (threshold={threshold:.2f})")
    print(f"{'═'*65}")
    print(f"  Accuracy     : {acc:.1f}%")
    print(f"  Sensitivity  : {sens:.1f}%   ← CP correctly detected")
    print(f"  Specificity  : {spec:.1f}%   ← Healthy correctly detected")
    print(f"  Precision    : {prec:.1f}%")
    print(f"  F1 Score     : {f1:.1f}")
    print(f"  MCC          : {mcc:.1f}")
    print(f"\n  Confusion Matrix:")
    print(f"                   Pred Healthy   Pred CP")
    print(f"  True Healthy :      {tn:>5}          {fp:>3}")
    print(f"  True CP      :      {fn:>5}          {tp:>3}")
    print(f"{'═'*65}")
    return dict(acc=acc,sens=sens,spec=spec,prec=prec,f1=f1,mcc=mcc,
                tp=int(tp),tn=int(tn),fp=int(fp),fn=int(fn))


def tune_threshold(labels, probs):
    print(f"\n  {'Threshold':>10}  {'Acc':>6}  {'Sens':>6}  {'Spec':>6}  "
          f"{'F1':>6}  {'TP':>4}  {'FP':>4}  {'FN':>4}")
    print(f"  {'-'*60}")

    best = {'f1': -1, 'thr': 0.5, 'sens': 0}
    best_sens = {'sens': -1, 'thr': 0.5}

    for thr in np.arange(0.1, 0.95, 0.05):
        p = (probs >= thr).astype(int)
        if len(np.unique(p)) < 2: continue
        tn,fp,fn,tp = confusion_matrix(labels, p, labels=[0,1]).ravel()
        acc  = (tp+tn)/len(labels)*100
        sens = tp/(tp+fn)*100  if (tp+fn) > 0 else 0.
        spec = tn/(tn+fp)*100  if (tn+fp) > 0 else 0.
        prec = tp/(tp+fp)*100  if (tp+fp) > 0 else 0.
        f1   = 2*prec*sens/(prec+sens) if (prec+sens) > 0 else 0.
        marker = ' ◄' if f1 > best['f1'] else ''
        print(f"  thr={thr:.2f}    {acc:>6.1f}%  {sens:>6.1f}%  {spec:>6.1f}%  "
              f"{f1:>6.1f}  {int(tp):>4}  {int(fp):>4}  {int(fn):>4}{marker}")
        if f1 > best['f1']:
            best = {'f1':f1,'thr':thr,'acc':acc,'sens':sens,'spec':spec}
        if sens > best_sens['sens']:
            best_sens = {'sens':sens,'thr':thr,'spec':spec}

    return best, best_sens


print(f"\n[5/5] Ensemble Results (Weighted: LSTM×0.3 + GRU×0.7):")
print("\n  All Fold Prediction Details:")
print(f"  {'Video':>6}  {'True':>8}  {'LSTM Prob':>10}  {'GRU Prob':>9}  {'Ens Prob':>9}  {'Result':>8}")
print(f"  {'-'*60}")
DISPLAY_THR = 0.35   # sensitivity-focused display threshold
for i in range(len(labels)):
    pred_ens = 1 if ens_probs[i] >= DISPLAY_THR else 0
    ok = '✓ OK' if pred_ens==labels[i] else '✗ WRONG'
    print(f"  {i+1:>6}  {'CP' if labels[i]==1 else 'Healthy':>8}  "
          f"{lstm_probs[i]:>10.4f}  {gru_probs[i]:>9.4f}  {ens_probs[i]:>9.4f}  {ok:>8}")

# Default threshold (0.5)
compute_and_print_metrics(labels, ens_probs, 0.5,  "Weighted Ensemble (thr=0.50)")

# Sensitivity-focused threshold
# compute_and_print_metrics(labels, ens_probs, 0.35, "Weighted Ensemble (thr=0.35) ← recommended")

# Threshold tuning (full sweep)
print("\n  Full Threshold Sweep:")
best_f1, best_sens = tune_threshold(labels, ens_probs)

# Save results
out_file = os.path.join(SAVE_DIR, 'bilstm_bigru_ensemble_results.npz')
np.savez(
    out_file,
    ens_probs=ens_probs, lstm_probs=lstm_probs, gru_probs=gru_probs, labels=labels,
    lstm_weight=0.4, gru_weight=0.6,
    recommended_thr=0.35,
    best_f1_thr=best_f1['thr'], best_sens_thr=best_sens['thr']
)

print(f"\n  ✓ Ensemble Results saved to: {out_file}")
print("="*65)
print("  DONE!")
print("="*65)

  [KAGGLE DETECTED] Scanning /kaggle/input recursively...
  ✓ Found sequences at: /kaggle/input/datasets/kareemyasser09/cp22222222222222/New folder/sliding_window_sequences_448.npz
  ✓ Found labels at: /kaggle/input/datasets/kareemyasser09/cp22222222222222/New folder/RVI_38_labels.mat
  ✓ Outputs will be saved to: /kaggle/working
  CP Detection Ensemble: BiLSTM + BiGRU + Temporal Attention
  TF version: 2.19.0
  ✓ GPU found (2 device(s)). Memory growth enabled.

[1/5] Loading sequences...
  Loaded 38 sequences
  Labels: CP=6 | Healthy=32
  Sequence lengths: min=150 max=574
  Feature dim: 448

  Fixed Class Weights (based on original 38 videos):
    Healthy (0) → 0.5938
    CP      (1) → 3.1667

[2/5] Data Augmentation strategy configured.

[3/5] Ensemble Architectures (BiLSTM & BiGRU) loaded.

[4/5] Running Improved Ensemble LOOCV...

═════════════════════════════════════════════════════════════════
  Fold 01/38  |  Test = Video 1  |  Label = Healthy
═══════════════════════════════════